# 🔢 Embeddings para TensorFlow Embedding Projector

Este notebook genera archivos TSV compatibles con [TensorFlow Embedding Projector](https://projector.tensorflow.org/) a partir de la base de conocimiento de Pink Panther.

**Dataset**: [Historias Pink Panther en Kaggle](https://www.kaggle.com/datasets/gustavodelacruztovar/historias-pink-panther)

Se incluyen **dos ejemplos**:

1. **Embeddings por chunks** — Vectoriza fragmentos de texto de los documentos
2. **Embeddings por palabra** — Vectoriza cada palabra única del corpus

## ¿Qué es Embedding Projector?

Es una herramienta web de TensorFlow que permite visualizar vectores de alta dimensión (embeddings) en 3D usando técnicas como:
- **PCA** (Principal Component Analysis)
- **t-SNE** (t-distributed Stochastic Neighbor Embedding)
- **UMAP** (Uniform Manifold Approximation and Projection)

## Modelo de Embeddings: all-MiniLM-L6-v2

- Produce vectores de **384 dimensiones**
- Muy rápido (~14,000 oraciones/segundo en GPU)
- Tamaño: ~80 MB
- Buen balance calidad/velocidad

---
## 📦 Instalación de dependencias

In [ ]:
!uv pip install --system -q sentence-transformers numpy kaggle

---
## 📂 Descargar el dataset de Kaggle

Usamos el CLI de Kaggle para descargar el dataset **Historias Pink Panther**.

Este dataset es público y de solo lectura, no requiere autenticación.

In [ ]:
# Descargar el dataset
!kaggle datasets download -d gustavodelacruztovar/historias-pink-panther --unzip -p kbpink

print("\n✅ Dataset descargado exitosamente")

In [ ]:
# Verificar que los archivos están en su lugar
import glob
md_files = sorted(glob.glob("kbpink/*.md"))
print(f"📄 Archivos encontrados en kbpink/: {len(md_files)}")
for f in md_files:
    print(f"   - {f}")

---
---
# 📘 EJEMPLO 1: Embeddings por Chunks

Divide los documentos en fragmentos (chunks) de ~1000 caracteres y genera un embedding por cada fragmento.

**Útil para**: Ver cómo se agrupan los temas/secciones de los documentos en el espacio vectorial.

In [ ]:
import os
import glob
import csv
import numpy as np
from sentence_transformers import SentenceTransformer

### 1.1 Funciones de carga y chunking

In [ ]:
def load_markdown_documents(directory: str) -> list:
    """Carga todos los archivos .md del directorio especificado."""
    documents = []
    md_files = sorted(glob.glob(os.path.join(directory, "*.md")))

    for filepath in md_files:
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        filename = os.path.basename(filepath)
        documents.append({
            "content": content,
            "filename": filename,
            "filepath": filepath,
        })
    print(f"📄 Cargados {len(documents)} documentos desde {directory}")
    return documents


def chunk_document(doc: dict, chunk_size: int = 1000, overlap: int = 200) -> list:
    """
    Divide un documento en fragmentos (chunks) más pequeños.

    - chunk_size: Tamaño máximo de cada chunk en caracteres
    - overlap: Caracteres que se solapan entre chunks consecutivos
    - Intenta cortar en saltos de párrafo para no romper ideas
    """
    text = doc["content"]
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]

        if end < len(text):
            last_newline = chunk_text.rfind("\n\n")
            if last_newline > chunk_size // 2:
                chunk_text = chunk_text[:last_newline]
                end = start + last_newline

        chunks.append({
            "content": chunk_text.strip(),
            "filename": doc["filename"],
            "chunk_id": f"{doc['filename']}_chunk_{len(chunks)}",
        })
        start = end - overlap if end < len(text) else len(text)

    return chunks

### 1.2 Cargar documentos y generar chunks

In [ ]:
# Cargar documentos
documents = load_markdown_documents("kbpink")

# Generar chunks
all_chunks = []
for doc in documents:
    chunks = chunk_document(doc)
    all_chunks.extend(chunks)

print(f"🔪 Generados {len(all_chunks)} chunks de texto")
print(f"\n📋 Ejemplo de chunk:")
print(f"   ID: {all_chunks[0]['chunk_id']}")
print(f"   Archivo: {all_chunks[0]['filename']}")
print(f"   Texto (primeros 200 chars): {all_chunks[0]['content'][:200]}...")

### 1.3 Generar embeddings con SentenceTransformer

In [ ]:
# Cargar modelo de embeddings
print("🧠 Cargando modelo de embeddings (all-MiniLM-L6-v2)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generar embeddings para todos los chunks
texts = [chunk["content"] for chunk in all_chunks]
print(f"📐 Generando embeddings para {len(texts)} chunks...")
embeddings_chunks = embedding_model.encode(texts, show_progress_bar=True)

print(f"\n✅ Embeddings generados: shape = {embeddings_chunks.shape}")
print(f"   → {embeddings_chunks.shape[0]} vectores de {embeddings_chunks.shape[1]} dimensiones cada uno")

### 1.4 Exportar a TSV para Embedding Projector

In [ ]:
OUTPUT_DIR = "projector_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Archivo de embeddings (vectores) ---
embeddings_path = os.path.join(OUTPUT_DIR, "embeddings.tsv")
with open(embeddings_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter="\t")
    for vector in embeddings_chunks:
        writer.writerow(vector.tolist())

# --- Archivo de metadata ---
metadata_path = os.path.join(OUTPUT_DIR, "metadata.tsv")
with open(metadata_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["chunk_id", "filename", "preview"])
    for chunk in all_chunks:
        preview = chunk["content"][:100].replace("\n", " ").replace("\t", " ")
        writer.writerow([chunk["chunk_id"], chunk["filename"], preview])

print(f"✅ Archivos generados:")
print(f"   📊 Vectores:  {embeddings_path}")
print(f"   📋 Metadata:  {metadata_path}")
print(f"   📏 {embeddings_chunks.shape[0]} vectores × {embeddings_chunks.shape[1]} dimensiones")

### 1.5 Descargar archivos TSV (chunks)

In [ ]:
from google.colab import files

print("📥 Descargando embeddings.tsv...")
files.download("projector_output/embeddings.tsv")

print("📥 Descargando metadata.tsv...")
files.download("projector_output/metadata.tsv")

---
---
# 📗 EJEMPLO 2: Embeddings por Palabra

Extrae todas las palabras únicas del corpus y genera un embedding por cada palabra individual.

**Útil para**: Ver cómo se agrupan las palabras por significado semántico (sinónimos cerca, antónimos lejos).

In [ ]:
import re
from collections import defaultdict

### 2.1 Extraer palabras únicas del corpus

In [ ]:
MIN_WORD_LENGTH = 4  # Filtrar palabras muy cortas (el, la, de, que...)

word_info = defaultdict(lambda: {"docs": set(), "count": 0})

for doc in documents:
    # Extraer palabras: solo letras (incluye acentos, ñ, etc.)
    words = re.findall(r"[a-záéíóúüñA-ZÁÉÍÓÚÜÑ]+", doc["content"].lower())
    for word in words:
        if len(word) >= MIN_WORD_LENGTH:
            word_info[word]["docs"].add(doc["filename"])
            word_info[word]["count"] += 1

unique_words = sorted(word_info.keys())
print(f"🔤 Palabras únicas extraídas: {len(unique_words)}")
print(f"   (mínimo {MIN_WORD_LENGTH} caracteres por palabra)")
print(f"\n📋 Primeras 20 palabras: {unique_words[:20]}")
print(f"📋 Últimas 20 palabras: {unique_words[-20:]}")

### 2.2 Generar embeddings por palabra

In [ ]:
# Reutilizamos el modelo ya cargado (embedding_model)
print(f"📐 Generando embeddings para {len(unique_words)} palabras...")
embeddings_words = embedding_model.encode(unique_words, show_progress_bar=True, batch_size=128)

print(f"\n✅ Embeddings generados: shape = {embeddings_words.shape}")
print(f"   → {embeddings_words.shape[0]} palabras × {embeddings_words.shape[1]} dimensiones")

### 2.3 Exportar a TSV para Embedding Projector

In [ ]:
# --- Archivo de embeddings (vectores) ---
embeddings_path_w = os.path.join(OUTPUT_DIR, "embeddings_palabras.tsv")
with open(embeddings_path_w, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter="\t")
    for vector in embeddings_words:
        writer.writerow(vector.tolist())

# --- Archivo de metadata ---
metadata_path_w = os.path.join(OUTPUT_DIR, "metadata_palabras.tsv")
with open(metadata_path_w, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(["word", "count", "num_docs", "sources"])
    for word in unique_words:
        info = word_info[word]
        count = info["count"]
        num_docs = len(info["docs"])
        sources_str = " | ".join(sorted(info["docs"]))
        writer.writerow([word, count, num_docs, sources_str])

print(f"✅ Archivos generados:")
print(f"   📊 Vectores:  {embeddings_path_w}")
print(f"   📋 Metadata:  {metadata_path_w}")
print(f"   📏 {embeddings_words.shape[0]} palabras × {embeddings_words.shape[1]} dimensiones")

### 2.4 Descargar archivos TSV (palabras)

In [ ]:
from google.colab import files

print("📥 Descargando embeddings_palabras.tsv...")
files.download("projector_output/embeddings_palabras.tsv")

print("📥 Descargando metadata_palabras.tsv...")
files.download("projector_output/metadata_palabras.tsv")

---
---
# 🔍 BONUS: Explorar similitud entre palabras

Antes de ir al Embedding Projector, podemos explorar la similitud entre palabras directamente aquí.

In [ ]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    """Calcula la similitud coseno entre dos vectores."""
    return np.dot(a, b) / (norm(a) * norm(b))

def find_similar_words(target_word, words_list, embeddings_matrix, top_n=10):
    """Encuentra las N palabras más similares a la palabra objetivo."""
    if target_word not in words_list:
        print(f"❌ '{target_word}' no está en el vocabulario")
        return

    idx = words_list.index(target_word)
    target_embedding = embeddings_matrix[idx]

    similarities = []
    for i, word in enumerate(words_list):
        if i != idx:
            sim = cosine_similarity(target_embedding, embeddings_matrix[i])
            similarities.append((word, sim))

    similarities.sort(key=lambda x: x[1], reverse=True)

    print(f"\n🔍 Top {top_n} palabras más similares a '{target_word}':")
    print(f"{'─'*40}")
    for word, sim in similarities[:top_n]:
        bar = '█' * int(sim * 20)
        print(f"   {word:<20} {sim:.4f} {bar}")

# Probar con algunas palabras interesantes
find_similar_words("amor", unique_words, embeddings_words)
find_similar_words("música", unique_words, embeddings_words)
find_similar_words("tecnología", unique_words, embeddings_words)

---
## 🌐 Cómo visualizar en Embedding Projector

1. Ir a **https://projector.tensorflow.org/**
2. Click en **"Load"** (panel izquierdo)
3. **Step 1**: Cargar el archivo `embeddings.tsv` o `embeddings_palabras.tsv`
4. **Step 2**: Cargar el archivo `metadata.tsv` o `metadata_palabras.tsv`
5. Explorar con **PCA**, **t-SNE** o **UMAP**

### Tips de visualización:
- **t-SNE** es mejor para ver clusters locales (grupos de palabras similares)
- **PCA** es mejor para ver la estructura global
- **UMAP** combina lo mejor de ambos
- Usa el buscador para encontrar palabras específicas
- Click en un punto para ver sus vecinos más cercanos